# Tentative de notebook pour le projet d'optimisation

## Importations

In [1]:
using JuMP
using HiGHS
using XLSX

## Lecture des données

In [4]:
data_file = "data/Donnees_etude_de_cas_ETE305.xlsx"

load = XLSX.readdata(data_file, "Résumé", "H2:H193")
offshore_load_factor = XLSX.readdata(data_file, "Résumé", "J2:J193")  
onshore_load_factor = XLSX.readdata(data_file, "Résumé", "I2:I193")
solar_load_factor = XLSX.readdata(data_file, "Résumé", "K2:K193")
hydro_fatal = XLSX.readdata(data_file, "Résumé", "L2:L193")
Pres = hydro_fatal 
thermique_fatal = XLSX.readdata(data_file, "Résumé", "M2:M193")

# data per week for hydro use (reference)
Usable_per_week_hydro_lacs= XLSX.readdata(data_file, "Résumé", "D2") #quantité d'hydro disponible à la semaine (en MWh) pour les lacs
Usable_per_week_hydro_STEP= XLSX.readdata(data_file, "Résumé", "D4") #quantité d'hydro disponible à la semaine (en MWh) pour les STEP

#initial capacities 
CapaSolar_init = XLSX.readdata(data_file, "Parc électrique", "C24") #MW
CapaOffshore_init = XLSX.readdata(data_file, "Parc électrique", "C23") #MW
CapaOnshore_init = XLSX.readdata(data_file, "Parc électrique", "C22") #MW

#### Loading of CAPEX/OPEX data
types_centrales=["onshore", "offshore_pose", "offshore_flot", "pv_pose", "pv_gd_toit", "pv_pet_toit", "CCG_H2", "TAC_H2", "electrolyseur", "batterie"]
CAPEX=XLSX.readdata(data_file, "Investissements", "B2:B11") # CAPEX des différentes technologies en €/kW
OPEX=XLSX.readdata(data_file, "Investissements", "C2:C11") # OPEX des différentes technologies en €/kW/an
Duree_vie=XLSX.readdata(data_file, "Investissements", "D2:D11") # Durée de vie des différentes technologies en années

#data for h2 clusters
capex_CCG_H2 = CAPEX[7]*1000 #€/MW
opex_CCG_H2 = OPEX[7]*1000 #€/MW/year
PU_cost_h2_CCG = XLSX.readdata(data_file, "Parc électrique", "H9") #€/MWh basé sur le tarif de prod de la centrale CCG gaz A MODIFIER
Pmin_h2_CCG = XLSX.readdata(data_file, "Parc électrique", "F9") #MW idem
Pmax_h2_CCG = XLSX.readdata(data_file, "Parc électrique", "E9") #MW idem
dmin_CCG = XLSX.readdata(data_file, "Parc électrique", "G9") #hours idem

capex_TAC_H2 = CAPEX[8]*1000 #€/MW
opex_TAC_H2 = OPEX[8]*1000 #€/MW/year
PU_cost_h2_TAC = XLSX.readdata(data_file, "Parc électrique", "H10") #€/MWh basé sur le tarif de prod de la centrale TAC gaz A MODIFIER
Pmin_h2_TAC = XLSX.readdata(data_file, "Parc électrique", "F10") #MW idem
Pmax_h2_TAC = XLSX.readdata(data_file, "Parc électrique", "E10") #MW idem
dmin_TAC = XLSX.readdata(data_file, "Parc électrique", "G10") #hours idem


# NH2_max_CCG = 10

# Pmin_h2_CCG = Pmin_h2_CCG*ones(NH2_max_CCG)
# Pmax_h2_CCG = Pmax_h2_CCG*ones(NH2_max_CCG)
# dmin_CCG = dmin_CCG*ones(Int, NH2_max_CCG)

NH2_max = 10

Pmin_h2 = Pmin_h2_TAC*ones(NH2_max)
Pmax_h2 = Pmax_h2_TAC*ones(NH2_max)
dmin = dmin_TAC*ones(Int, NH2_max)


#data for hydro reservoir "lacs"
Nhy = 1 #number of hydro generation units
Pmin_hy_lacs = zeros(Nhy)
Pmax_hy_lacs = XLSX.readdata(data_file, "Parc électrique", "C20") *ones(Nhy) #MW
e_hy_lacs = Usable_per_week_hydro_lacs*ones(Nhy) #MWh


# variable costs
cuns = XLSX.readdata(data_file, "Defaillance", "B2") #cost of unsupplied energy €/MWh
cexc = XLSX.readdata(data_file, "Defaillance", "B3") #cost of in excess energy €/MWh

 #investment costs 
capex_onshore = CAPEX[1]*1000 #€/MW
capex_offshore = CAPEX[2]*1000 #€/MW
capex_solar = CAPEX[4]*1000 #€/MW
capex_2h_battery = CAPEX[10]*1000 #€/MW

opex_onshore = OPEX[1]*1000 #€/MW
opex_offshore = OPEX[2]*1000 #€/MW
opex_solar = OPEX[4]*1000 #€/MW
opex_2h_battery = OPEX[10]*1000 #€/MW


#data for STEP/battery
#weekly STEP
Pmax_STEP = XLSX.readdata(data_file, "Parc électrique", "C21") #MW
rSTEP = XLSX.readdata(data_file, "Rendements", "B10") #rendement

#battery
rbattery = XLSX.readdata(data_file, "Rendements", "B9") # rendement de la batterie au sockage ou au destockage
d_battery = XLSX.readdata(data_file, "Investissements", "E11") #hours
CapaBattery_init = 0 #MW

0

In [7]:
Tweek = 168 #optimization for 1 week (7*24=168 hours)
Tguard = 24
NH2_max = 10

Tmax = Tweek + Tguard

192

In [5]:
model = Model(HiGHS.Optimizer)

A JuMP Model
├ solver: HiGHS
├ objective_sense: FEASIBILITY_SENSE
├ num_variables: 0
├ num_constraints: 0
└ Names registered in the model: none

In [11]:
#thermal generation variables
@variable(model, Pth[1:Tmax,1:NH2_max] >= 0)
@variable(model, UCth[1:Tmax,1:NH2_max], Bin)
@variable(model, UPth[1:Tmax,1:NH2_max], Bin)
@variable(model, DOth[1:Tmax,1:NH2_max], Bin)
#hydro generation variables
@variable(model, Phy[1:Tmax,1:Nhy] >= 0)
#unsupplied energy variables
@variable(model, Puns[1:Tmax] >= 0)
#in excess energy variables
@variable(model, Pexc[1:Tmax] >= 0)
#weekly STEP variables
@variable(model, Pcharge_STEP[1:Tmax] >= 0)
@variable(model, Pdecharge_STEP[1:Tmax] >= 0)
@variable(model, stock_STEP[1:Tmax] >= 0)
# #battery variables
@variable(model, Pcharge_battery[1:Tmax] >= 0)
@variable(model, Pdecharge_battery[1:Tmax] >= 0)
@variable(model, stock_battery[1:Tmax] >= 0)

ErrorException: An object of name Pth is already attached to this model. If this
    is intended, consider using the anonymous construction syntax, for example,
    `x = @variable(model, [1:N], ...)` where the name of the object does
    not appear inside the macro.

    Alternatively, use `unregister(model, :Pth)` to first unregister
    the existing name from the model. Note that this will not delete the
    object; it will just remove the reference at `model[:Pth]`.


In [ ]:
@objective(model, Min, 
                        CapaSolar*(capex_solar + opex_solar) + CapaOffshore*(capex_offshore + opex_offshore) + CapaOnshore*(capex_onshore + opex_onshore) 
                        + CapaBattery*(capex_2h_battery + opex_2h_battery) 
                        + sum(H2installed[g]*Pmax_h2[g] for g in 1:NH2_max)*(capex_CCG_H2 + opex_CCG_H2) + sum(PH2[t,g] for t in 1:Tmax, g in 1:NH2_max)*PU_cost_h2_CCG
                        + sum(Puns[t] for t in 1:Tmax)*cuns + sum(Pexc[t] for t in 1:Tmax)*cexc
)

In [ ]:
#balance constraint
@constraint(model, balance[t in 1:Tmax], sum(PH2[t,g] for g in 1:NH2_max) + sum(Phy[t,h] for h in 1:Nhy) + CapaSolar * solar_load_factor[t] + CapaOffshore * offshore_load_factor[t] + CapaOnshore * onshore_load_factor[t] + Puns[t] - load[t] - Pexc[t] - Pcharge_STEP[t] + Pdecharge_STEP[t] - Pcharge_battery[t] + Pdecharge_battery[t] == 0)

# H2 Power constraints
@constraint(model, max_H2[t in 1:Tmax, i in 1:NH2_max], PH2[t,i] <= Pmax_h2[i]*H2running[t,i]) #Pmax constraints
@constraint(model, min_H2[t in 1:Tmax, i in 1:NH2_max], Pmin_h2[i]*H2running[t,i] <= PH2[t,i]) #Pmin constraints

# H2 duration constraints
for g in 1:NH2_max
        if (dmin[g] > 1)
            @constraint(model, [t in 2:Tmax], H2running[t,g]-H2running[t-1,g]==H2start[t,g]-H2stop[t,g],  base_name = "stateH2_$g") # detect start and stop
            @constraint(model, [t in 1:Tmax], H2start[t,g]+H2stop[t,g]<=1,  base_name = "exclusiveH2_$g") # avoid starting and stoping at same step

            # Initial conditions
            @constraint(model, H2start[1,g]==0,  base_name = "iniStartH2_$g")
            @constraint(model, H2stop[1,g]==0,  base_name = "iniStopH2_$g")
            @constraint(model, [t in 1:dmin[g]-1], H2running[t,g] >= sum(H2start[i,g] for i in 1:t), base_name = "dminStartH2_$(g)_init")
            @constraint(model, [t in 1:dmin[g]-1], H2running[t,g] <= 1-sum(H2stop[i,g] for i in 1:t), base_name = "dminStopH2_$(g)_init")

            # Minimum up and down time constraints
            @constraint(model, [t in dmin[g]:Tmax], H2running[t,g] >= sum(H2start[i,g] for i in (t-dmin[g]+1):t),  base_name = "dminStartH2_$g")
            @constraint(model, [t in dmin[g]:Tmax], H2running[t,g] <= 1 - sum(H2stop[i,g] for i in (t-dmin[g]+1):t),  base_name = "dminStopH2_$g")
end

# H2 volume constraints
RendementElectrolyse = 0.7 # Rendement de l'électrolyse
RendementCombustion = 0.5 # Rendement de la combustion de l'hydrogène
@constraint(model, volume_H2, sum(PH2[t,g] for t in 1:Tmax, g in 1:NH2_max) <= sum(Pexc[t] for t in 1:Tmax)*RendementCombustion*RendementElectrolyse)

# hydro unit constraints
@constraint(model, bounds_hy[t in 1:Tmax, h in 1:Nhy], Pmin_hy[h] <= Phy[t,h] <= Pmax_hy[h])
# hydro stock constraint
@constraint(model, stock_hy[h in 1:Nhy], sum(Phy[t,h] for t in 1:Tmax) <= e_hy_week[h])

# weekly STEP
@constraint(model, Pcharge_max_STEP[t in 1:Tmax], Pcharge_STEP[t] <= Pmax_STEP)
@constraint(model, Pdecharge_max_STEP[t in 1:Tmax], Pdecharge_STEP[t] <= Pmax_STEP)
@constraint(model, init_stock_STEP, stock_STEP[1] == 0)
@constraint(model, end_Pdecharge_STEP, Pdecharge_STEP[Tmax] <= stock_STEP[Tmax])
@constraint(model, Tmax_stock_STEP, stock_STEP[Tmax] == stock_STEP[1])
@constraint(model, init_Pdecharge_STEP, Pdecharge_STEP[1] == 0)
@constraint(model, evol_stock_STEP[t in 1:Tmax-1], stock_STEP[t+1]-stock_STEP[t]- rSTEP*Pcharge_STEP[t]+Pdecharge_STEP[t]== 0)
@constraint(model, stock_max_STEP[t in 1:Tmax], stock_STEP[t] <= 24*7*Pmax_STEP)

# #battery
@constraint(model, Pcharge_max_battery[t in 1:Tmax], Pcharge_battery[t] <= CapaBattery)
@constraint(model, Pdecharge_max_battery[t in 1:Tmax], Pdecharge_battery[t] <= CapaBattery)
@constraint(model, init_stock_battery, stock_battery[1] == 0)
@constraint(model, end_Pdecharge_battery, Pdecharge_battery[Tmax] <= stock_battery[Tmax])
@constraint(model, Tmax_stock_battery, stock_battery[Tmax] == stock_battery[1])
@constraint(model, init_Pdecharge_battery, Pdecharge_battery[1] == 0)
@constraint(model, evol_stock_battery[t in 1:Tmax-1], stock_battery[t+1]-stock_battery[t]- rbattery*Pcharge_battery[t]+1/rbattery*Pdecharge_battery[t]== 0)
@constraint(model, stock_max_battery[t in 1:Tmax], stock_battery[t] <= d_battery*CapaBattery)
